In [14]:
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from data.loader import BaseLoader
from tqdm import tqdm

In [3]:
class RealDataLoader(BaseLoader):
    """
    Loads real market data from a directory of per-ticker CSV files.

    Expects one CSV per ticker, named {ticker}.csv, each containing
    at minimum a 'timestamp' column and a 'close' column. Loads the
    full ticker universe present in the directory; cross-dataset
    alignment (intersection with synthetic tickers) is handled
    downstream, not here.

    Attributes:

        directory : str
            Path to the folder containing the CSV files.

    Example::

        loader  = RealDataLoader(directory="/path/to/raw_intraday")
        dataset = loader.load()
    """

    def __init__(self, directory: str):
        super().__init__(name="Real", is_synthetic=False)
        self.directory = directory

    def _load_raw(self) -> pd.DataFrame:
        csv_files = sorted([f for f in os.listdir(self.directory) if f.endswith(".csv")])

        if not csv_files:
            raise FileNotFoundError(f"No CSV files found in {self.directory}")

        series_list = []
        skipped = []
        for filename in tqdm(csv_files, desc=f"Loading {self.name}"):
            ticker = filename.replace(".csv", "")
            path = os.path.join(self.directory, filename)

            cols = pd.read_csv(path, nrows=0).columns.tolist()
            if "timestamp" not in cols or "close" not in cols:
                skipped.append(ticker)
                continue

            df = pd.read_csv(
                path, usecols=["timestamp", "close"], index_col="timestamp", parse_dates=True
            )
            series_list.append(df["close"].rename(ticker))

        if skipped:
            print(
                f"Skipped {len(skipped)} files (missing 'timestamp' or 'close'): "
                f"{skipped[:10]}{'...' if len(skipped) > 10 else ''}"
            )

        prices = pd.concat(series_list, axis=1)
        prices.index = pd.to_datetime(prices.index, utc=True)
        prices = prices.sort_index()

        return prices

In [4]:
class AILSyntheticLoader(BaseLoader):
    """
    Loads AIL synthetic data from a long-format parquet file.

    Expects columns: tr_ric, timestamp, close. Tickers in tr_ric
    carry exchange suffixes that are stripped to plain symbols to
    match the real data's convention. Loads the full ticker universe
    present in the file; cross-dataset alignment is handled downstream.

    Attributes:

        parquet_path : str
            Path to the AIL synthetic parquet file.

    Example::

        loader  = AILSyntheticLoader(parquet_path="path/to/ail.parquet")
        dataset = loader.load()
    """

    def __init__(self, parquet_path: str):
        super().__init__(name="AIL", is_synthetic=True)
        self.parquet_path = parquet_path

    def _load_raw(self) -> pd.DataFrame:
        print("Reading parquet...")
        table = pq.read_table(self.parquet_path, columns=["tr_ric", "timestamp", "close"])
        raw = table.to_pandas()

        print("Stripping RIC suffixes...")
        # Strip any trailing exchange suffix (.N, .OQ, .A, .K, ...) generically
        raw["ticker"] = raw["tr_ric"].str.replace(r"\.\w+$", "", regex=True)

        print("Pivoting to wide format...")
        prices = raw.pivot_table(index="timestamp", columns="ticker", values="close")
        prices.index = pd.to_datetime(prices.index).tz_localize("UTC")
        prices = prices.sort_index()

        print(f"Done: {prices.shape}")
        return prices

In [5]:
real = RealDataLoader(
    directory="/home/rishabh/PycharmProjects/SyntheticGenerators/data/raw_intraday"
).load()
ail = AILSyntheticLoader(
    parquet_path="/home/rishabh/PycharmProjects/SyntheticGenerators/data/ail_synthetic_data/"
    "dataset_US_1-10B_2019-09-2020-03.parquet"
).load()

print(f"Real tickers: {real.prices.shape[1]}")
print(f"AIL tickers:  {ail.prices.shape[1]}")

intersection = sorted(set(real.prices.columns) & set(ail.prices.columns))
print(f"Intersection: {len(intersection)}")

real_prices = real.prices[intersection]
ail_prices = ail.prices[intersection]

Loading Real: 100%|██████████| 949/949 [02:43<00:00,  5.80it/s]


Skipped 1 files (missing 'timestamp' or 'close'): ['download_log']


/tmp/ipykernel_6333/2685035245.py:53: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  prices = pd.concat(series_list, axis=1)


Reading parquet...
Stripping RIC suffixes...
Pivoting to wide format...
Done: (106822, 952)
Real tickers: 948
AIL tickers:  952
Intersection: 948


In [6]:
real_prices

,AA,AAL,AAN,AAP,AAT,AAXN,ABCB,ABM,ABMD,ACA,...,XRX,YELP,YETI,YEXT,YY,ZEN,ZG,ZION,ZNGA,ZS
timestamp,,,,,,,,,,,,,,,,,,,,,
2019-09-03 13:31:00+00:00,17.56,26.130,254.72,136.7700,NaN,58.42,NaN,36.98,NaN,NaN,...,28.8400,33.11,27.1400,15.5650,56.865,79.33,34.03,40.57,5.615,67.1712
2019-09-03 13:32:00+00:00,17.43,26.045,255.80,138.1000,NaN,58.26,NaN,NaN,190.950,NaN,...,28.7512,33.08,27.1500,15.5401,56.837,NaN,33.99,40.52,5.645,67.5400
2019-09-03 13:33:00+00:00,17.42,26.110,255.24,137.2300,46.74,58.55,NaN,36.83,190.930,NaN,...,28.7000,33.06,27.5000,15.5600,56.700,79.77,NaN,40.56,5.655,67.7050
2019-09-03 13:34:00+00:00,17.35,26.100,255.80,137.2025,NaN,58.64,NaN,NaN,190.930,NaN,...,28.7800,33.09,27.4200,15.6500,56.760,79.92,33.79,40.58,5.690,67.8900
2019-09-03 13:35:00+00:00,17.35,26.070,254.48,137.4650,NaN,58.62,34.83,NaN,190.805,NaN,...,28.8000,33.21,27.5772,15.6850,56.750,80.15,33.96,40.56,5.690,68.4700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-03-20 19:59:00+00:00,5.47,10.380,59.96,75.0300,22.38,60.78,20.85,21.05,130.180,31.12,...,16.4000,17.80,16.3100,10.8400,42.070,56.89,26.34,25.58,5.950,53.6400
2020-03-20 20:00:00+00:00,NaN,10.310,NaN,NaN,NaN,60.78,20.76,21.05,130.510,31.16,...,16.4000,NaN,NaN,NaN,42.250,56.88,26.49,25.58,6.210,53.6400
2020-03-20 20:01:00+00:00,NaN,10.480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,42.020,NaN,NaN,NaN,5.950,NaN


In [7]:
ail_prices

ticker,AA,AAL,AAN,AAP,AAT,AAXN,ABCB,ABM,ABMD,ACA,...,XRX,YELP,YETI,YEXT,YY,ZEN,ZG,ZION,ZNGA,ZS
timestamp,,,,,,,,,,,,,,,,,,,,,
2019-09-03 08:46:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-09-03 09:06:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-09-03 09:55:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-09-03 11:01:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-09-03 11:15:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,56.84,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-03-20 23:56:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-03-20 23:57:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-03-20 23:58:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
import pandas_market_calendars as mcal

nyse = mcal.get_calendar("XNYS")

# Schedule across the sample period
schedule = nyse.schedule(start_date="2019-09-03", end_date="2020-03-20")

# Minute-level clock; mcal returns tz-aware UTC timestamps, DST handled
market_clock = mcal.date_range(schedule, frequency="1min")

# date_range gives session *closes* at minute marks; inspect the boundaries
print(f"Clock length: {len(market_clock)}")
print(f"First: {market_clock[0]}")
print(f"Last:  {market_clock[-1]}")
print(f"Timezone: {market_clock.tz}")

Clock length: 53850
First: 2019-09-03 13:31:00+00:00
Last:  2020-03-20 20:00:00+00:00
Timezone: UTC


In [9]:
def session_bounds(prices, ticker, dates):
    sub = prices[ticker].dropna()
    for d in dates:
        day = sub[sub.index.date == pd.Timestamp(d).date()]
        if len(day):
            print(f"  {d}: first={day.index[0].time()}, last={day.index[-1].time()} UTC")


check_dates = ["2019-10-15", "2019-11-01", "2019-11-04", "2019-12-02"]

print("REAL session bounds (UTC) across DST boundary:")
session_bounds(real_prices, intersection[0], check_dates)

print("\nAIL session bounds (UTC) across DST boundary:")
session_bounds(ail_prices, intersection[0], check_dates)

REAL session bounds (UTC) across DST boundary:
  2019-10-15: first=13:18:00, last=23:26:00 UTC
  2019-11-01: first=09:51:00, last=20:04:00 UTC
  2019-11-04: first=12:44:00, last=21:28:00 UTC
  2019-12-02: first=12:00:00, last=23:29:00 UTC

AIL session bounds (UTC) across DST boundary:
  2019-10-15: first=13:32:00, last=20:04:00 UTC
  2019-11-01: first=13:33:00, last=20:04:00 UTC
  2019-11-04: first=14:32:00, last=21:04:00 UTC
  2019-12-02: first=14:31:00, last=21:03:00 UTC


In [10]:
# Reindex real onto the regular-session clock
real_aligned = real_prices.reindex(market_clock)

# How much of real survives (i.e. lands on regular-session minutes)?
real_coverage = real_aligned.notna().mean()
print("Real coverage on regular-session clock:")
print(real_coverage.describe())

# Sanity: how many real bars were OUTSIDE the clock (extended hours, dropped)?
total_real_bars = real_prices.notna().sum().sum()
on_clock_bars = real_aligned.notna().sum().sum()
print(f"\nReal bars total:        {total_real_bars:,}")
print(f"Real bars on clock:     {on_clock_bars:,}")
print(f"Real bars dropped (ext-hours / off-grid): {total_real_bars - on_clock_bars:,}")
print(f"Fraction dropped: {(total_real_bars - on_clock_bars) / total_real_bars:.1%}")

Real coverage on regular-session clock:
count    948.000000
mean       0.823627
std        0.151471
min        0.096007
25%        0.723273
50%        0.859666
75%        0.952642
max        0.998960
dtype: float64

Real bars total:        42,891,090
Real bars on clock:     42,046,011
Real bars dropped (ext-hours / off-grid): 845,079
Fraction dropped: 2.0%


In [11]:
ail_aligned = ail_prices.reindex(market_clock)

# Restrict both to the intersection universe
real_cov = real_aligned[intersection].notna().mean()
ail_cov = ail_aligned[intersection].notna().mean()

coverage = pd.DataFrame({"real": real_cov, "ail": ail_cov})
print("Coverage on regular-session clock (intersection tickers):")
print(coverage.describe())

# Joint coverage: minutes where BOTH have data (relevant for any per-minute logic,
# though metrics are distributional)
print(f"\nTickers where real coverage < 50%: {(coverage['real'] < 0.50).sum()}")
print(f"Tickers where ail coverage  < 50%: {(coverage['ail'] < 0.50).sum()}")
print(f"Tickers where real coverage >= 80%: {(coverage['real'] >= 0.80).sum()}")
print(f"Tickers where ail coverage  >= 80%: {(coverage['ail'] >= 0.80).sum()}")

Coverage on regular-session clock (intersection tickers):
             real         ail
count  948.000000  948.000000
mean     0.823627    0.748039
std      0.151471    0.118868
min      0.096007    0.167874
25%      0.723273    0.661792
50%      0.859666    0.757205
75%      0.952642    0.842358
max      0.998960    0.926054

Tickers where real coverage < 50%: 26
Tickers where ail coverage  < 50%: 8
Tickers where real coverage >= 80%: 595
Tickers where ail coverage  >= 80%: 359


In [12]:
coverage["joint_min"] = coverage[["real", "ail"]].min(axis=1)

print("Joint minimum coverage (worse of the two sides):")
print(coverage["joint_min"].describe())

for floor in [0.50, 0.60, 0.70, 0.75, 0.80]:
    n = (coverage["joint_min"] >= floor).sum()
    print(f"  Floor {floor:.0%}: {n} tickers survive")

Joint minimum coverage (worse of the two sides):
count    948.000000
mean       0.737388
std        0.129440
min        0.096007
25%        0.648918
50%        0.750019
75%        0.839169
max        0.926054
Name: joint_min, dtype: float64
  Floor 50%: 920 tickers survive
  Floor 60%: 809 tickers survive
  Floor 70%: 600 tickers survive
  Floor 75%: 474 tickers survive
  Floor 80%: 339 tickers survive


In [15]:
# Provisional universe
floor = 0.70
universe = coverage.index[coverage["joint_min"] >= floor].tolist()
print(f"Universe at {floor:.0%} floor: {len(universe)} tickers")

real_u = real_aligned[universe]
ail_u = ail_aligned[universe]

# Per-minute missingness rate across the universe (fraction of tickers missing each minute)
real_miss_rate = real_u.isna().mean(axis=1)
ail_miss_rate = ail_u.isna().mean(axis=1)

# Daily aggregation for readability
real_miss_daily = real_miss_rate.groupby(real_miss_rate.index.date).mean()
ail_miss_daily = ail_miss_rate.groupby(ail_miss_rate.index.date).mean()

# A crude market-volatility proxy: cross-sectional dispersion of 1-min returns per day,
# computed on REAL data (the ground truth for "was this a turbulent day")
real_ret = np.log(real_u).diff()
daily_vol = (
    real_ret.std(axis=1).groupby(real_ret.index.date).mean()
)  # avg cross-sectional vol per day

mnar = pd.DataFrame(
    {
        "real_miss": real_miss_daily,
        "ail_miss": ail_miss_daily,
        "market_vol": daily_vol,
    }
)
mnar.index = pd.to_datetime(mnar.index)

# Correlation: does missingness rise with volatility?
print("\nCorrelation of daily missingness with market volatility:")
print(f"  Real missingness vs vol: {mnar['real_miss'].corr(mnar['market_vol']):.3f}")
print(f"  AIL  missingness vs vol: {mnar['ail_miss'].corr(mnar['market_vol']):.3f}")

# Zoom on the crash window specifically
crash = mnar.loc["2020-02-20":"2020-03-20"]
calm = mnar.loc[:"2020-02-19"]
print(f"\nMean AIL missingness — calm period:  {calm['ail_miss'].mean():.3f}")
print(f"Mean AIL missingness — crash period: {crash['ail_miss'].mean():.3f}")
print(f"Mean real missingness — calm period:  {calm['real_miss'].mean():.3f}")
print(f"Mean real missingness — crash period: {crash['real_miss'].mean():.3f}")

Universe at 70% floor: 600 tickers

Correlation of daily missingness with market volatility:
  Real missingness vs vol: -0.575
  AIL  missingness vs vol: -0.573

Mean AIL missingness — calm period:  0.200
Mean AIL missingness — crash period: 0.073
Mean real missingness — calm period:  0.106
Mean real missingness — crash period: 0.043
